# 02 Exploratory Data Analysis & Statistical Validation

## Portfolio Project: Customer Retention & Revenue Risk Analytics

### Overview
This notebook calculates overall business KPIs, evaluates customer churn across contract types, tenure groups, payment methods, internet services, and demographics, and validates relationships using Chi-Square tests and two-sample hypothesis tests.

---


In [ ]:
import sys, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sys.path.append(r'c:\Users\Atul\Desktop\project-2\customer-retention-revenue-risk-analytics')

from src.feature_engineering import engineer_features

proc_path = r'c:\Users\Atul\Desktop\project-2\customer-retention-revenue-risk-analytics\data\processed\cleaned_telco_churn.csv'
df_proc = pd.read_csv(proc_path)
df_eng = engineer_features(df_proc)

print(f"Dataset Shape: {df_eng.shape}")


### Executive Baseline KPIs
Calculating Total Customers, Active vs Churned, Churn Rate (%), Average Monthly Charges, Average Tenure, Total MRR, and Historical Monthly Revenue Loss.


In [ ]:
total_cust = len(df_eng)
churned_cust = (df_eng['Churn_Numeric'] == 1).sum()
active_cust = total_cust - churned_cust
churn_rate = (churned_cust / total_cust) * 100
avg_monthly = df_eng['MonthlyCharges'].mean()
avg_tenure = df_eng['Tenure'].mean()
total_mrr = df_eng['MonthlyCharges'].sum()
mrr_lost = df_eng[df_eng['Churn_Numeric'] == 1]['MonthlyCharges'].sum()

kpi_summary = pd.DataFrame({
    'KPI Metric': [
        'Total Customers', 'Active Customers', 'Churned Customers',
        'Overall Churn Rate (%)', 'Avg Monthly Charges ($)',
        'Avg Customer Tenure (Months)', 'Total Monthly Revenue (MRR) ($)',
        'Monthly Revenue Lost from Churn ($)'
    ],
    'Value': [
        f"{total_cust:,}", f"{active_cust:,}", f"{churned_cust:,}",
        f"{churn_rate:.2f}%", f"${avg_monthly:.2f}",
        f"{avg_tenure:.1f}", f"${total_mrr:,.2f}", f"${mrr_lost:,.2f}"
    ]
})
kpi_summary


### Customer Segmentation Analysis
Segmenting by Tenure Group, Monthly Charges Revenue Tier, and Contract Type.


In [ ]:
# 1. Tenure Group Segmentation
tenure_seg = df_eng.groupby('TenureGroup', observed=False).agg(
    TotalCustomers=('CustomerID', 'count'),
    ChurnedCustomers=('Churn_Numeric', 'sum'),
    AvgMonthlyCharges=('MonthlyCharges', 'mean'),
    TotalMonthlyRevenue=('MonthlyCharges', 'sum')
).reset_index()
tenure_seg['ChurnRatePct'] = (tenure_seg['ChurnedCustomers'] / tenure_seg['TotalCustomers']) * 100
print("--- Tenure Group Breakdown ---")
display(tenure_seg)

# 2. Contract Type Segmentation
contract_seg = df_eng.groupby('Contract').agg(
    TotalCustomers=('CustomerID', 'count'),
    ChurnedCustomers=('Churn_Numeric', 'sum'),
    AvgMonthlyCharges=('MonthlyCharges', 'mean'),
    MonthlyRevenueLost=('MonthlyCharges', lambda x: x[df_eng.loc[x.index, 'Churn_Numeric'] == 1].sum())
).reset_index()
contract_seg['ChurnRatePct'] = (contract_seg['ChurnedCustomers'] / contract_seg['TotalCustomers']) * 100
print("\n--- Contract Type Breakdown ---")
display(contract_seg)


### Statistical Significance Hypothesis Testing
Conducting Chi-Square tests for categorical features and Mann-Whitney U / T-tests for numerical variables.


In [ ]:
# 1. Chi-Square Test: Contract Type vs Churn
contingency_contract = pd.crosstab(df_eng['Contract'], df_eng['Churn'])
chi2_c, p_c, dof_c, _ = stats.chi2_contingency(contingency_contract)
print(f"Chi-Square Test (Contract vs Churn): Chi2 = {chi2_c:.2f}, p-value = {p_c:.4e}")

# 2. Chi-Square Test: Tech Support vs Churn
contingency_tech = pd.crosstab(df_eng['TechSupport'], df_eng['Churn'])
chi2_t, p_t, dof_t, _ = stats.chi2_contingency(contingency_tech)
print(f"Chi-Square Test (TechSupport vs Churn): Chi2 = {chi2_t:.2f}, p-value = {p_t:.4e}")

# 3. Two-Sample T-test: Monthly Charges for Churned vs Active Customers
monthly_churned = df_eng[df_eng['Churn_Numeric'] == 1]['MonthlyCharges']
monthly_active = df_eng[df_eng['Churn_Numeric'] == 0]['MonthlyCharges']
t_stat, p_val_m = stats.ttest_ind(monthly_churned, monthly_active)
print(f"T-test (MonthlyCharges Churned vs Active): t-stat = {t_stat:.2f}, p-value = {p_val_m:.4e}")

# 4. Mann-Whitney U test: Tenure for Churned vs Active Customers
tenure_churned = df_eng[df_eng['Churn_Numeric'] == 1]['Tenure']
tenure_active = df_eng[df_eng['Churn_Numeric'] == 0]['Tenure']
u_stat, p_val_u = stats.mannwhitneyu(tenure_churned, tenure_active)
print(f"Mann-Whitney U Test (Tenure Churned vs Active): U-stat = {u_stat:.2f}, p-value = {p_val_u:.4e}")
